# Notebook 5: Motif Visual Interpretability and Probe Analysis

This notebook is the image-first interpretation surface for the cached motif artifacts from `nb03`, plus the full linear probe analysis from `z` to class labels.

It compares `phase_b.pt` and `phase_c.pt` side by side and never retrains or rediscover motifs.


In [ ]:
# Colab-first setup: clone/update the repo, install it, and mount Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Running outside Colab; using the current working tree.")


In [ ]:
from flow_circuits.data import CIFAR10_STATS, build_cifar10_splits
from flow_circuits.evaluation import (
    NB05_EXPERIMENT_IDS,
    run_linear_probe_suite_experiment,
    run_motif_case_study_experiment,
    run_motif_visual_report_experiment,
    run_phase_motif_comparison_experiment,
    run_probe_confusion_analysis_experiment,
    run_probe_error_analysis_experiment,
)
from flow_circuits.training import collect_interpretability_outputs, load_components_from_checkpoint


## Config

Change the single `EXPERIMENTS` line below to run all six `nb05` experiments or any subset.
This notebook requires cached motif-family artifacts from `nb03` and optionally uses `nb04` phase-match outputs when present.


In [ ]:
from pathlib import Path

RUN_MODE = "full"  # "debug" or "full"
CONFIG_NAME = "resnet18_aligned"
EXPERIMENTS = "all"
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits")
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / CONFIG_NAME
NB03_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_recurring_motif_core_validation" / CONFIG_NAME
NB04_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb04_motif_extended_characterization" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb05_motif_visual_interpretability_and_probe_analysis" / CONFIG_NAME

PHASE_B_CHECKPOINT = EXPERIMENT_ROOT / "phase_b.pt"
PHASE_C_CHECKPOINT = EXPERIMENT_ROOT / "phase_c.pt"


In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch

try:
    import pandas as pd
except ImportError:
    pd = None

MODEL_ORDER = [("phase_b", "Phase B"), ("phase_c", "Phase C")]
EXPERIMENT_LABELS = {
    "motif_reports": "Motif Reports",
    "phase_comparison": "Phase Comparison",
    "intervention_cases": "Intervention Cases",
    "class_probe_suite": "Class Probe Suite",
    "confusion_analysis": "Confusion Analysis",
    "error_analysis": "Error Analysis",
}
CIFAR10_LABELS = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

if EXPERIMENTS == "all":
    SELECTED_EXPERIMENTS = list(NB05_EXPERIMENT_IDS)
else:
    SELECTED_EXPERIMENTS = [str(name) for name in EXPERIMENTS]
invalid = sorted(set(SELECTED_EXPERIMENTS) - set(NB05_EXPERIMENT_IDS))
if invalid:
    raise ValueError(f"Unknown experiments: {invalid}. Valid ids: {NB05_EXPERIMENT_IDS}")

if not PHASE_B_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_B_CHECKPOINT}")
if not PHASE_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_C_CHECKPOINT}")

REQUIRED_NB03_ARTIFACTS = {
    "phase_b": NB03_OUTPUT_DIR / "phase_b" / "motif_families.json",
    "phase_c": NB03_OUTPUT_DIR / "phase_c" / "motif_families.json",
}
for tag, artifact_path in REQUIRED_NB03_ARTIFACTS.items():
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing required motif artifact from nb03: {artifact_path}")

OPTIONAL_NB04_PHASE_MATCH = NB04_OUTPUT_DIR / "comparisons" / "motif_phase_match.json"
OPTIONAL_NB03_RESULTS = {
    tag: {
        "motif_predictiveness": NB03_OUTPUT_DIR / tag / "motif_predictiveness.json",
        "motif_interventions": NB03_OUTPUT_DIR / tag / "motif_interventions.json",
    }
    for tag, _label in MODEL_ORDER
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR = OUTPUT_DIR / "comparisons"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_BY_MODE = {
    "debug": {
        "motif_reports": {"max_images": 1000, "topk": 4, "min_visual_budget": 4, "exemplar_count": 9},
        "phase_comparison": {},
        "intervention_cases": {"max_images": 1000, "topk": 2, "exemplar_count": 4, "alpha": 0.05},
        "class_probe_suite": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500},
        "confusion_analysis": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "exemplar_count": 6},
        "error_analysis": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "exemplar_count": 6},
    },
    "full": {
        "motif_reports": {"max_images": 5000, "topk": 6, "min_visual_budget": 6, "exemplar_count": 9},
        "phase_comparison": {},
        "intervention_cases": {"max_images": 5000, "topk": 4, "exemplar_count": 6, "alpha": 0.05},
        "class_probe_suite": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000},
        "confusion_analysis": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "exemplar_count": 6},
        "error_analysis": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "exemplar_count": 6},
    },
}
if RUN_MODE not in SETTINGS_BY_MODE:
    raise ValueError(f"RUN_MODE must be one of {sorted(SETTINGS_BY_MODE)}")


def _format_seconds(seconds):
    if seconds is None:
        return "?"
    seconds = int(max(0, round(seconds)))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


def _progress_logger(**event):
    stamp = time.strftime("%H:%M:%S")
    label = {"phase_b": "Phase B", "phase_c": "Phase C", "comparisons": "Compare"}.get(event["checkpoint_tag"], event["checkpoint_tag"])
    stage = event["stage"].replace("_", " ")
    total = event.get("total")
    counter = stage if total is None else f"{stage} {event['completed']}/{total}"
    elapsed = _format_seconds(event.get("elapsed_seconds"))
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    if eta is None:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | {message}", flush=True)
    else:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | ETA {_format_seconds(eta)} | {message}", flush=True)


def _cache_path(tag, experiment_id):
    return OUTPUT_DIR / tag / f"{experiment_id}.json"


def _load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _run_or_load(tag, experiment_id, run_fn):
    cache_path = _cache_path(tag, experiment_id)
    if cache_path.exists() and not FORCE_RERUN:
        print(f"Loading cached {experiment_id} for {tag}: {cache_path}")
        return _load_json(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    result = run_fn(cache_path)
    return _load_json(cache_path) if cache_path.exists() else result


def _should_run(experiment_id):
    return experiment_id in SELECTED_EXPERIMENTS


def _write_summary_table(rows, *, title):
    print(title)
    if not rows:
        print("  No rows to display.")
        return
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)


def _class_name(idx):
    if idx is None:
        return None
    idx = int(idx)
    return CIFAR10_LABELS[idx] if 0 <= idx < len(CIFAR10_LABELS) else str(idx)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"Run mode      : {RUN_MODE}")
print(f"Config        : {CONFIG_NAME}")
print(f"Experiments   : {SELECTED_EXPERIMENTS}")
print(f"Phase B ckpt  : {PHASE_B_CHECKPOINT}")
print(f"Phase C ckpt  : {PHASE_C_CHECKPOINT}")
print(f"nb03 root     : {NB03_OUTPUT_DIR}")
print(f"nb04 root     : {NB04_OUTPUT_DIR}")
print(f"Output dir    : {OUTPUT_DIR}")

components_by_tag = {
    "phase_b": load_components_from_checkpoint(PHASE_B_CHECKPOINT, device=device),
    "phase_c": load_components_from_checkpoint(PHASE_C_CHECKPOINT, device=device),
}
base_config = components_by_tag["phase_b"].config
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=int(base_config["data"]["batch_size"]),
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)

RESULTS = {}
MOTIF_ARTIFACTS = {tag: _load_json(path) for tag, path in REQUIRED_NB03_ARTIFACTS.items()}
NB04_PHASE_MATCH = _load_json(OPTIONAL_NB04_PHASE_MATCH) if OPTIONAL_NB04_PHASE_MATCH.exists() else None
DISCOVERY_OUTPUTS = {}
TEST_OUTPUTS = {}


def _optional_nb03_result(tag, experiment_id):
    path = OPTIONAL_NB03_RESULTS[tag][experiment_id]
    return _load_json(path) if path.exists() else None


def _get_outputs(tag, split, max_images):
    store = DISCOVERY_OUTPUTS if split == "discovery" else TEST_OUTPUTS
    if tag in store and store[tag]["images"].shape[0] >= max_images:
        return store[tag]
    outputs = collect_interpretability_outputs(
        components_by_tag[tag],
        loaders[split],
        device=device,
        max_images=max_images,
    )
    store[tag] = outputs
    return outputs


def _denormalize_image(image_tensor):
    mean = torch.tensor(CIFAR10_STATS["mean"]).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STATS["std"]).view(3, 1, 1)
    image = image_tensor.detach().cpu() * std + mean
    image = image.clamp(0.0, 1.0)
    return image.permute(1, 2, 0).numpy()


def _draw_overlay(ax, overlay):
    if overlay is None:
        return
    for box in overlay.get("active_boxes", []):
        edgecolor = "tab:red" if box.get("is_representative") else "tab:cyan"
        rect = patches.Rectangle((box["x0"], box["y0"]), box["x1"] - box["x0"], box["y1"] - box["y0"], linewidth=2, edgecolor=edgecolor, facecolor="none")
        ax.add_patch(rect)


def _show_cards(cards, outputs, title):
    print(title)
    for card in cards:
        print(f"  {card['object_type']} {card['id']} | class={card['dominant_class_name']} | purity={card['class_purity']:.3f} | stability={card['stability']:.3f}")
        if not card.get("exemplar_row_indices"):
            continue
        cols = min(3, len(card["exemplar_row_indices"]))
        rows = int(math.ceil(len(card["exemplar_row_indices"]) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
        axes = np.atleast_1d(axes).reshape(rows, cols)
        for ax in axes.flat:
            ax.axis("off")
        for ax, row_idx, dataset_idx, label in zip(axes.flat, card["exemplar_row_indices"], card.get("exemplar_dataset_indices", []), card.get("exemplar_labels", [])):
            ax.imshow(_denormalize_image(outputs["images"][int(row_idx)]))
            _draw_overlay(ax, card.get("overlay"))
            ax.set_title(f"idx={dataset_idx} | label={_class_name(label)}")
        plt.show()


def _show_examples(examples, outputs, title):
    print(title)
    if not examples:
        print("  No examples to display.")
        return
    cols = min(3, len(examples))
    rows = int(math.ceil(len(examples) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.flat:
        ax.axis("off")
    for ax, example in zip(axes.flat, examples):
        ax.imshow(_denormalize_image(outputs["images"][int(example["row_index"])]))
        if "overlay" in example:
            _draw_overlay(ax, example["overlay"])
        title_bits = []
        for key in ("label_name", "true_label_name", "pred_label_name", "backbone_pred_name"):
            if key in example:
                title_bits.append(f"{key.replace('_name','')}={example[key]}")
        if "delta_margin" in example:
            title_bits.append(f"Δ={example['delta_margin']:.3f}")
        if "probe_true_margin" in example:
            title_bits.append(f"probe={example['probe_true_margin']:.3f}")
        ax.set_title(" | ".join(title_bits))
    plt.show()


## Experiment 1: Motif Reports

Render motif cards with exemplar images and active-cell overlays.

In [ ]:
if _should_run("motif_reports"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_reports"]
    RESULTS["motif_reports"] = {
        tag: _run_or_load(
            tag,
            "motif_reports",
            lambda cache_path, tag=tag: run_motif_visual_report_experiment(
                components_by_tag[tag],
                loaders["discovery"],
                device=device,
                checkpoint_tag=tag,
                motif_artifact=MOTIF_ARTIFACTS[tag],
                max_images=settings["max_images"],
                topk=settings["topk"],
                min_visual_budget=settings["min_visual_budget"],
                exemplar_count=settings["exemplar_count"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping motif_reports.")


In [ ]:
if "motif_reports" in RESULTS:
    _write_summary_table([
        {"checkpoint": label, **RESULTS["motif_reports"][tag]["summary"]}
        for tag, label in MODEL_ORDER
    ], title="Motif report summary")
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_reports"]
    for tag, label in MODEL_ORDER:
        outputs = _get_outputs(tag, "discovery", settings["max_images"])
        _show_cards(RESULTS["motif_reports"][tag]["visual_objects"][:3], outputs, f"{label} motif visuals")


## Experiment 2: Phase Comparison

Compare the strongest Phase B and Phase C motif views side by side, using `nb04` phase matches when present.

In [ ]:
if _should_run("phase_comparison"):
    if "motif_reports" not in RESULTS:
        raise RuntimeError("phase_comparison requires motif_reports in the current notebook run")
    RESULTS["phase_comparison"] = _run_or_load(
        "comparisons",
        "phase_comparison",
        lambda cache_path: run_phase_motif_comparison_experiment(
            RESULTS["motif_reports"]["phase_b"],
            RESULTS["motif_reports"]["phase_c"],
            phase_match_artifact=NB04_PHASE_MATCH,
            output_path=cache_path,
        ),
    )
else:
    print("Skipping phase_comparison.")


In [ ]:
if "phase_comparison" in RESULTS:
    _write_summary_table([RESULTS["phase_comparison"]["summary"]], title="Phase comparison summary")
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_reports"]
    phase_b_outputs = _get_outputs("phase_b", "discovery", settings["max_images"])
    phase_c_outputs = _get_outputs("phase_c", "discovery", settings["max_images"])
    for pair in RESULTS["phase_comparison"]["comparison_rows"][:2]:
        _show_cards([pair["phase_b"]], phase_b_outputs, "Phase B comparison card")
        _show_cards([pair["phase_c"]], phase_c_outputs, "Phase C comparison card")


## Experiment 3: Intervention Cases

Show the strongest held-out member images before and after motif ablation.

In [ ]:
if _should_run("intervention_cases"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["intervention_cases"]
    RESULTS["intervention_cases"] = {
        tag: _run_or_load(
            tag,
            "intervention_cases",
            lambda cache_path, tag=tag: run_motif_case_study_experiment(
                components_by_tag[tag],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                motif_artifact=MOTIF_ARTIFACTS[tag],
                max_images=settings["max_images"],
                topk=settings["topk"],
                exemplar_count=settings["exemplar_count"],
                alpha=settings["alpha"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping intervention_cases.")


In [ ]:
if "intervention_cases" in RESULTS:
    _write_summary_table([
        {"checkpoint": label, **RESULTS["intervention_cases"][tag]["summary"]}
        for tag, label in MODEL_ORDER
    ], title="Intervention case summary")
    settings = SETTINGS_BY_MODE[RUN_MODE]["intervention_cases"]
    for tag, label in MODEL_ORDER:
        outputs = _get_outputs(tag, "test", settings["max_images"])
        for case_row in RESULTS["intervention_cases"][tag]["case_studies"][:2]:
            _show_examples(case_row["exemplars"], outputs, f"{label} motif {case_row['motif_id']} case study")


## Experiment 4: Class Probe Suite

Fit global, per-layer, and per-node linear probes from `z` to class labels.

In [ ]:
if _should_run("class_probe_suite"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["class_probe_suite"]
    RESULTS["class_probe_suite"] = {
        tag: _run_or_load(
            tag,
            "class_probe_suite",
            lambda cache_path, tag=tag: run_linear_probe_suite_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                val_max_images=settings["val_max_images"],
                test_max_images=settings["test_max_images"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping class_probe_suite.")


In [ ]:
if "class_probe_suite" in RESULTS:
    _write_summary_table([
        {
            "checkpoint": label,
            **RESULTS["class_probe_suite"][tag]["summary"],
        }
        for tag, label in MODEL_ORDER
    ], title="Class probe summary")
    for tag, label in MODEL_ORDER:
        _write_summary_table(RESULTS["class_probe_suite"][tag]["top_nodes_by_coef_norm"], title=f"{label} top probe nodes")


## Experiment 5: Confusion Analysis

Anchor on the backbone's hardest confusion pairs and ask where `z` contains the most linearly separable information for those pairs.

In [ ]:
if _should_run("confusion_analysis"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["confusion_analysis"]
    RESULTS["confusion_analysis"] = {
        tag: _run_or_load(
            tag,
            "confusion_analysis",
            lambda cache_path, tag=tag: run_probe_confusion_analysis_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                val_max_images=settings["val_max_images"],
                test_max_images=settings["test_max_images"],
                top_pairs=settings["top_pairs"],
                exemplar_count=settings["exemplar_count"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping confusion_analysis.")


In [ ]:
if "confusion_analysis" in RESULTS:
    _write_summary_table([
        {"checkpoint": label, **RESULTS["confusion_analysis"][tag]["summary"]}
        for tag, label in MODEL_ORDER
    ], title="Confusion analysis summary")
    settings = SETTINGS_BY_MODE[RUN_MODE]["confusion_analysis"]
    for tag, label in MODEL_ORDER:
        pair_rows = RESULTS["confusion_analysis"][tag]["pair_rows"]
        if pair_rows:
            _write_summary_table(pair_rows[0]["top_nodes"], title=f"{label} top nodes for hardest pair")
            outputs = _get_outputs(tag, "test", settings["test_max_images"])
            _show_examples(pair_rows[0]["exemplar_errors"], outputs, f"{label} hardest confusion exemplars")


## Experiment 6: Error Analysis

Compare probe margins on backbone-correct vs backbone-wrong images to see where useful class information disappears.

In [ ]:
if _should_run("error_analysis"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["error_analysis"]
    RESULTS["error_analysis"] = {
        tag: _run_or_load(
            tag,
            "error_analysis",
            lambda cache_path, tag=tag: run_probe_error_analysis_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                val_max_images=settings["val_max_images"],
                test_max_images=settings["test_max_images"],
                exemplar_count=settings["exemplar_count"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping error_analysis.")


In [ ]:
if "error_analysis" in RESULTS:
    _write_summary_table([
        {"checkpoint": label, **RESULTS["error_analysis"][tag]["summary"]}
        for tag, label in MODEL_ORDER
    ], title="Error analysis summary")
    settings = SETTINGS_BY_MODE[RUN_MODE]["error_analysis"]
    for tag, label in MODEL_ORDER:
        _write_summary_table(RESULTS["error_analysis"][tag]["top_drop_nodes"], title=f"{label} top margin-drop nodes")
        outputs = _get_outputs(tag, "test", settings["test_max_images"])
        _show_examples(RESULTS["error_analysis"][tag]["failure_examples"], outputs, f"{label} failure examples")


## Final Summary

Compact checkpoint comparison across all selected `nb05` experiments.

In [ ]:
comparison_rows = []
if "motif_reports" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["motif_reports"], "phase_b": RESULTS["motif_reports"]["phase_b"]["summary"]["mean_class_purity"], "phase_c": RESULTS["motif_reports"]["phase_c"]["summary"]["mean_class_purity"], "metric": "mean_class_purity"})
if "phase_comparison" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["phase_comparison"], "phase_b": RESULTS["phase_comparison"]["summary"]["mean_phase_b_purity"], "phase_c": RESULTS["phase_comparison"]["summary"]["mean_phase_c_purity"], "metric": "mean_compared_purity"})
if "intervention_cases" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["intervention_cases"], "phase_b": RESULTS["intervention_cases"]["phase_b"]["summary"]["validated_count"], "phase_c": RESULTS["intervention_cases"]["phase_c"]["summary"]["validated_count"], "metric": "validated_count"})
if "class_probe_suite" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["class_probe_suite"], "phase_b": RESULTS["class_probe_suite"]["phase_b"]["summary"]["global_accuracy"], "phase_c": RESULTS["class_probe_suite"]["phase_c"]["summary"]["global_accuracy"], "metric": "global_accuracy"})
if "confusion_analysis" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["confusion_analysis"], "phase_b": RESULTS["confusion_analysis"]["phase_b"]["summary"]["mean_global_accuracy"], "phase_c": RESULTS["confusion_analysis"]["phase_c"]["summary"]["mean_global_accuracy"], "metric": "mean_global_pair_accuracy"})
if "error_analysis" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["error_analysis"], "phase_b": RESULTS["error_analysis"]["phase_b"]["summary"]["global_margin_drop"], "phase_c": RESULTS["error_analysis"]["phase_c"]["summary"]["global_margin_drop"], "metric": "global_margin_drop"})
_write_summary_table(comparison_rows, title="Motif interpretability and probe analysis summary")
(COMPARISON_DIR / "summary.json").write_text(json.dumps(comparison_rows, indent=2), encoding="utf-8")
